# Pack Run 8 TBS adjacency matrices (6 wells x 36 timepoints)

Reads the raw per-well, per-timepoint connectivity results produced by the
Super-Selective ECR pipeline for `Run_8_TBS_Experiment_ecr_results/` and
packs them into a single self-describing HDF5 file plus a companion
README, suitable for handing to someone outside this repo.

Source data: 36 whole-plate recordings (18 from `230601 RUN 8 Wells 1-3`,
18 from `230602 RUN 8 Wells 4-6`), each containing all 6 wells
(`well000`-`well005`).

In [1]:
import numpy as np
import pickle as pkl
import h5py
import os
import pandas as pd
from datetime import datetime

dr = '/ns/cis.jhu.edu/project/organoid/2024May28 No window data /OneDrive_1_6-17-2024/Run_8_TBS_Experiment_ecr_results/'
time_file = '/ns/cis.jhu.edu/project/organoid/April 19 2024/Time-file.csv'

out_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'packaged_data') if False else \
    '/cis/home/tchen94/tianyi/Organoid/Time course LTP(END)/packaged_data'
os.makedirs(out_dir, exist_ok=True)

h5_path = os.path.join(out_dir, 'Run8_TBS_adjacency_matrices.h5')
readme_path = os.path.join(out_dir, 'README_Run8_TBS_adjacency_matrices.md')

## 1. Build the ordered list of 36 source files

In [2]:
dr1_name = '230601 RUN 8 Wells 1-3'
dr2_name = '230602 RUN 8 Wells 4-6'

def sorted_folder_files(dr, subfolder):
    full = os.path.join(dr, subfolder)
    filenames = [f for f in os.listdir(full) if f != '.DS_Store']
    filenames = sorted(filenames, key=lambda x: int(x.split('#')[1].split()[0]))
    return [f'{subfolder}/{fn}/data.raw_20240521_16h16m.pkl' for fn in filenames]

files1 = sorted_folder_files(dr, dr1_name)
files2 = sorted_folder_files(dr, dr2_name)
all_files = files1 + files2

n_timepoints = len(all_files)
print(n_timepoints)
all_files

36


['230601 RUN 8 Wells 1-3/#1 (baseline)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#2 (well #1 post stim 1)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#3 (well #1 post stim 2)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#4 (well #1 post stim 3)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#5 (well #1 post stim 4)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#6 (well #2 post stim 1)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#7 (well #2 post stim 2)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#8 (well #2 post stim 3)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#9 (well #2 post stim 4)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#10 (well #3 post stim 1)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#11 (well #3 post stim 2)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#12 (well #3 post stim 3)/data.raw_20240521_16h16m.pkl',
 '230601 RUN 8 Wells 1-3/#13 (well #

## 2. Recording time of each file, minutes since the first (baseline) recording

In [3]:
df = pd.read_csv(time_file)
df['time'] = pd.to_datetime(df['Unnamed: 1'].astype(str) + ' ' + df['Unnamed: 2'].astype(str), errors='coerce')
for i in df.loc[df['Unnamed: 2'].isnull(), 'time'].index:
    df.loc[i, 'time'] = df.loc[i - 1, 'time'] + pd.Timedelta(10, 'min')

time = df[df['Unnamed: 2'].notna()]['time'].reset_index(drop=True)
time_since_baseline_min = np.array([(t - time[0]).total_seconds() / 60 for t in time])
assert len(time_since_baseline_min) == n_timepoints
time_since_baseline_min

array([   0.        ,   13.2       ,   26.43333333,   39.68333333,
         52.9       ,   71.03333333,   84.26666667,   97.5       ,
        110.75      ,  130.8       ,  144.05      ,  157.26666667,
        170.48333333,  190.05      ,  220.31666667,  250.18333333,
        279.85      ,  310.01666667, 1380.45      , 1393.66666667,
       1406.88333333, 1420.11666667, 1433.36666667, 1451.8       ,
       1465.08333333, 1478.36666667, 1491.65      , 1512.61666667,
       1525.86666667, 1539.1       , 1552.35      , 1572.48333333,
       1602.33333333, 1631.4       , 1662.11666667, 1691.5       ])

## 3. Link-filtering helper

Same filter used throughout the analysis notebooks: a predicted link is
dropped if every correlation-peak delay between that pair of channels was
smaller than one sampling period (`1/fs`), which flags links that are more
likely explained by a shared global event than a direct connection.

In [4]:
def filtered_adjacency(data, well):
    adj_matrix = data[well]['win_0']['adj_matrix_predicted']
    corr_peaks = data[well]['win_0']['corr_peaks']
    fs = data['config']['data']['fs']

    synced_matrix = np.full(adj_matrix.shape, False)
    for key in corr_peaks.keys():
        if np.all(np.abs(np.array(corr_peaks[key]['delays'])) < 1 / fs):
            synced_matrix[key[0], key[1]] = True
            synced_matrix[key[1], key[0]] = True

    return np.logical_and(adj_matrix, np.logical_not(synced_matrix))

## 4. Extract all 6 wells x 36 timepoints and write the HDF5 package

In [5]:
wells = [f'well{i:03d}' for i in range(6)]

with h5py.File(h5_path, 'w') as h5:
    h5.attrs['n_wells'] = len(wells)
    h5.attrs['n_timepoints'] = n_timepoints
    h5.attrs['created'] = datetime.now().isoformat()
    h5.attrs['source_dir'] = dr
    h5.attrs['description'] = (
        'Filtered directed adjacency matrices (boolean, links with '
        'globally-synchronized-only correlation peaks removed) for the '
        'Run 8 TBS experiment. 6 wells x 36 timepoints, whole-plate '
        'recordings, variable electrode count per well/timepoint.'
    )

    meta = h5.create_group('metadata')
    meta.create_dataset('source_filenames', data=np.array(all_files, dtype='S'))
    meta.create_dataset('time_since_baseline_min', data=time_since_baseline_min.astype('f8'))

    fs_seen = None
    for well in wells:
        g = h5.create_group(well)
        adj_g = g.create_group('adjacency')
        ch_g = g.create_group('channel_numbers')
        num_vertices = []

        for ti, fpath in enumerate(all_files):
            full = os.path.join(dr, fpath)
            with open(full, 'rb') as f:
                data = pkl.load(f)

            adj = filtered_adjacency(data, well)
            chn = np.asarray(data[well]['channel_numbers'], dtype='i8')
            fs = data['config']['data']['fs']
            fs_seen = fs

            adj_g.create_dataset(f't{ti:02d}', data=adj.astype(bool), compression='gzip')
            ch_g.create_dataset(f't{ti:02d}', data=chn)
            num_vertices.append(adj.shape[0])

        g.create_dataset('num_vertices', data=np.array(num_vertices, dtype='i4'))
        print(well, 'done -', min(num_vertices), '-', max(num_vertices), 'vertices across timepoints')

    h5.attrs['fs_hz'] = int(fs_seen)

print('wrote', h5_path)

/tmp/ipykernel_3486814/2561844960.py:29: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pkl.load(f)


well000 done - 613 - 651 vertices across timepoints


well001 done - 445 - 501 vertices across timepoints


well002 done - 348 - 411 vertices across timepoints


well003 done - 687 - 944 vertices across timepoints


well004 done - 940 - 987 vertices across timepoints


well005 done - 929 - 958 vertices across timepoints
wrote /cis/home/tchen94/tianyi/Organoid/Time course LTP(END)/packaged_data/Run8_TBS_adjacency_matrices.h5


## 5. Sanity-check the package by reading it back

In [6]:
with h5py.File(h5_path, 'r') as h5:
    print(dict(h5.attrs))
    print('wells:', list(h5.keys()))
    print('well003 num_vertices:', h5['well003/num_vertices'][:])
    print('well003 t00 adjacency shape:', h5['well003/adjacency/t00'].shape,
          h5['well003/adjacency/t00'].dtype)
    print('well003 t00 channel_numbers shape:', h5['well003/channel_numbers/t00'].shape)
    print('first 3 source filenames:', h5['metadata/source_filenames'][:3])
    print('first 5 time_since_baseline_min:', h5['metadata/time_since_baseline_min'][:5])

{'created': '2026-08-14T15:49:09.947364', 'description': 'Filtered directed adjacency matrices (boolean, links with globally-synchronized-only correlation peaks removed) for the Run 8 TBS experiment. 6 wells x 36 timepoints, whole-plate recordings, variable electrode count per well/timepoint.', 'fs_hz': np.int64(10000), 'n_timepoints': np.int64(36), 'n_wells': np.int64(6), 'source_dir': '/ns/cis.jhu.edu/project/organoid/2024May28 No window data /OneDrive_1_6-17-2024/Run_8_TBS_Experiment_ecr_results/'}
wells: ['metadata', 'well000', 'well001', 'well002', 'well003', 'well004', 'well005']
well003 num_vertices: [936 935 934 940 909 930 944 935 939 925 925 932 913 912 896 907 911 937
 931 922 920 913 687 926 918 929 919 928 922 935 941 934 913 932 903 909]
well003 t00 adjacency shape: (936, 936) bool
well003 t00 channel_numbers shape: (936,)
first 3 source filenames: [b'230601 RUN 8 Wells 1-3/#1 (baseline)/data.raw_20240521_16h16m.pkl'
 b'230601 RUN 8 Wells 1-3/#2 (well #1 post stim 1)/data

## 6. Write the companion README

In [7]:
readme_text = """# Run 8 TBS adjacency matrices

## What this is

Filtered connectivity graphs extracted from the Super-Selective ECR
pipeline output for the "Run 8 TBS" organoid MEA experiment
(`Run_8_TBS_Experiment_ecr_results/`, recorded 2023-06-01/02, processed
2024-05-21).

**6 wells x 36 timepoints = 216 directed graphs.**

- **6 wells**: `well000` ... `well005` (0-indexed; corresponds to wells 1-6
  as labeled on the physical MEA plate).
- **36 timepoints**: one whole-plate recording each, in chronological
  order, spanning two recording days:
  - timepoints 0-17: 2023-06-01, folder `230601 RUN 8 Wells 1-3`
  - timepoints 18-35: 2023-06-02, folder `230602 RUN 8 Wells 4-6`

  Every recording captures **all 6 wells simultaneously** -- the folder
  names ("Wells 1-3" / "Wells 4-6") and per-file labels
  (e.g. "well #4 post stim 3") only describe which well was being
  electrically stimulated and what elapsed time that implies for that
  well; they do not restrict which wells' data are present in the file.
  `metadata/source_filenames` in the HDF5 file gives the exact source
  file for each timepoint index if that stimulation context is needed.

## File format

Single HDF5 file: `Run8_TBS_adjacency_matrices.h5`

```
/                                          (root attrs: n_wells, n_timepoints,
                                             fs_hz, created, source_dir, description)
/metadata/
    source_filenames                       (36,) string - relative path of the
                                             source .pkl for each timepoint index
    time_since_baseline_min                (36,) float - minutes elapsed since
                                             the first (baseline) recording

/well000 .. /well005/
    num_vertices                           (36,) int - electrode/node count
                                             for that well at each timepoint
    adjacency/t00 .. t35                   (n_i, n_i) bool - directed adjacency
                                             matrix at timepoint i (see below)
    channel_numbers/t00 .. t35             (n_i,) int - electrode/channel ID
                                             for each row/column of that
                                             timepoint's adjacency matrix
```

**Why ragged, not one 3D array**: the number of active electrodes (and
which physical channels they are) differs per well and per timepoint, so
matrices are NOT padded to a common size and are NOT guaranteed to share
node identity across timepoints. Use `channel_numbers/tXX` to align nodes
across timepoints for the same well (e.g. before graph matching / MDS,
as done in the analysis notebooks) -- do not assume row i means the same
electrode at every timepoint.

## What "adjacency matrix" means here

- Square boolean matrix, `adj[i, j] = True` means a predicted directed
  functional link from electrode i to electrode j.
- Built from `adj_matrix_predicted` in the raw ECR output, then
  link-filtered: a link is removed if every correlation-peak delay
  between that channel pair was smaller than one sampling period
  (`1 / fs`), which flags links more likely explained by a shared
  global/simultaneous event than a direct connection. `fs` (sampling
  rate, Hz) is stored as a root-level HDF5 attribute.

## Minimal read example

```python
import h5py

with h5py.File("Run8_TBS_adjacency_matrices.h5", "r") as h5:
    adj = h5["well003/adjacency/t05"][:]      # (n, n) bool matrix, well003, timepoint 5
    ch = h5["well003/channel_numbers/t05"][:] # electrode IDs for that matrix's rows/cols
    t_min = h5["metadata/time_since_baseline_min"][5]  # minutes since baseline
```
"""

with open(readme_path, "w") as f:
    f.write(readme_text)

print("wrote", readme_path)

wrote /cis/home/tchen94/tianyi/Organoid/Time course LTP(END)/packaged_data/README_Run8_TBS_adjacency_matrices.md
